# Trening w Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MartaL-ctrl/Counting-crops/blob/main/notebooks/colab_train.ipynb)

**Jak używać:**
1. *Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU (np. T4)*.
2. W komórce **Ustawienia** wybierz plik konfiguracyjny (`CONFIG`).
3. *Środowisko wykonawcze → Uruchom wszystko*. Za pierwszym razem Colab poprosi o dostęp do Dysku Google.

**Gdzie co jest:**
- kod: GitHub → `/content/Counting-crops` (przy każdym uruchomieniu `git pull`),
- dane: archiwum `gwhd_2021.zip` z Zenodo (10,2 GB) na lokalnym dysku Colaba `/content/data/`
  (pobierane w każdej sesji albo kopiowane z Dysku Google, jeśli je tam zapiszesz),
- wyniki: Dysk Google `MyDrive/counting-crops/outputs/<run_name>/`.

**Po rozłączeniu sesji** uruchom wszystko ponownie. Jeśli na Dysku jest `last.pt`, trening wznowi się
od następnej epoki.

Kod zmieniamy lokalnie w VS Code (commit + push), a nie w Colabie: zmiany w `/content` znikają po sesji.

## Ustawienia

In [ ]:
CONFIG = "configs/colab_smoke.yaml"   # <- plik konfiguracyjny z repozytorium
RESUME = True                         # wznów od last.pt, jeśli istnieje

REPO_URL = "https://github.com/MartaL-ctrl/Counting-crops.git"
REPO_DIR = "/content/Counting-crops"
DRIVE_DIR = "/content/drive/MyDrive/counting-crops"   # wyniki (i opcjonalnie dane) na Dysku Google
LOCAL_ZIP = "/content/data/gwhd_2021.zip"             # musi się zgadzać z data_zip w konfiguracji

# Zbiór danych: oryginalne wydanie GWHD 2021 z Zenodo (DOI 10.5281/zenodo.5092309), wersja 1.0
ZENODO_URL = "https://zenodo.org/records/5092309/files/gwhd_2021.zip?download=1"
ZENODO_MD5 = "22b4b542c9ae7e056d7fcdeae9ecaed5"
ZENODO_SIZE = 10_215_190_132
# Zapis ZIP-a na Dysku Google oszczędza pobierania w kolejnych sesjach, ale zajmuje 10,2 GB
# z 15 GB darmowego Dysku - domyślnie wyłączony.
KEEP_ZIP_ON_DRIVE = False

## 1. Dysk Google

Daje Colabowi dostęp do Twojego Dysku jako folderu `/content/drive/MyDrive`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Kod z GitHuba

Pierwsze uruchomienie w sesji: `git clone`; kolejne: `git pull`. Wypisany skrót commita warto zanotować -
mówi, która wersja kodu dała dany wynik (zapisuje się też do `outputs/<run_name>/git_commit.txt`).

In [ ]:
import os
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
os.chdir(REPO_DIR)
print("commit:", COMMIT, "| katalog roboczy:", os.getcwd())

## 3. GPU i biblioteki

PyTorch, torchvision, pandas, pyarrow i PyYAML są w Colabie preinstalowane - **nie instalujemy ich ponownie**
(nadpisanie PyTorch może zepsuć obsługę GPU). Komórka tylko sprawdza wersje.

Jeśli `GPU: False` - zmień typ środowiska wykonawczego na GPU i uruchom wszystko od nowa.

In [ ]:
import sys

import pandas, pyarrow, torch, torchvision, yaml

print("Python", sys.version.split()[0])
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| pandas", pandas.__version__, "| pyarrow", pyarrow.__version__, "| pyyaml", yaml.__version__)
print("GPU:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
assert torch.cuda.is_available(), "Brak GPU - zmień typ środowiska wykonawczego"

## 4. Dane

Archiwum `gwhd_2021.zip` musi leżeć na lokalnym dysku Colaba (szybki odczyt; nie rozpakowujemy go -
obrazy są czytane prosto z ZIP-a). Kolejność:
1. jest już w `/content/data/` (ta sama sesja) - nic nie robi,
2. jest na Dysku Google (`MyDrive/counting-crops/data/`) - kopiuje,
3. w przeciwnym razie pobiera z Zenodo (10,2 GB, zwykle kilka minut) i sprawdza sumę MD5.

In [ ]:
import hashlib
import shutil
import subprocess
from pathlib import Path

local_zip = Path(LOCAL_ZIP)
drive_zip = Path(DRIVE_DIR) / "data" / "gwhd_2021.zip"
local_zip.parent.mkdir(parents=True, exist_ok=True)

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()

if local_zip.exists() and local_zip.stat().st_size == ZENODO_SIZE:
    print("ZIP jest już na dysku Colaba")
elif drive_zip.exists() and drive_zip.stat().st_size == ZENODO_SIZE:
    print("kopiowanie z Dysku Google...")
    shutil.copy2(drive_zip, local_zip)
else:
    print("pobieranie z Zenodo (10,2 GB)...")
    subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(local_zip), ZENODO_URL], check=True)
    digest = md5(local_zip)
    assert digest == ZENODO_MD5, f"zła suma MD5: {digest} (oczekiwano {ZENODO_MD5}) - usuń plik i pobierz ponownie"
    print("suma MD5 zgodna")
    if KEEP_ZIP_ON_DRIVE:
        drive_zip.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_zip, drive_zip)
        print("zapisano kopię na Dysku Google:", drive_zip)

assert local_zip.stat().st_size == ZENODO_SIZE, "niepełny plik ZIP"
print(f"{local_zip}: {local_zip.stat().st_size / 1e9:.2f} GB")

## 5. Trening

Wynik każdej epoki pojawia się na bieżąco. Jeśli `RESUME = True` i istnieje `last.pt`, trening jest wznawiany.

Colab rozłącza sesję po dłuższej bezczynności karty przeglądarki i po kilku godzinach pracy.
Checkpoint `last.pt` zapisuje się po każdej epoce, więc tracisz najwyżej jedną epokę.

In [ ]:
import yaml

with open(CONFIG, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
assert cfg["data_zip"] == LOCAL_ZIP, f"data_zip w {CONFIG} ({cfg['data_zip']}) != LOCAL_ZIP ({LOCAL_ZIP})"

run_dir = Path(cfg["output_dir"]) / cfg["run_name"]
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "git_commit.txt").write_text(COMMIT + "\n")

last = run_dir / "last.pt"
resume_args = f"--resume {last}" if RESUME and last.exists() else ""
print("wyniki:", run_dir, "|", "wznowienie od " + str(last) if resume_args else "nowy trening")

!python train.py --config {CONFIG} {resume_args}

## 6. Podgląd wyników

Pliki zostają na Dysku Google: `MyDrive/counting-crops/outputs/<run_name>/`.

In [ ]:
import pandas as pd

hist = pd.read_csv(run_dir / "history.csv")
display(hist)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["epoch"], hist["train_loss"])
ax[0].set(title="strata treningowa", xlabel="epoka", yscale="log")
ev = hist.dropna(subset=["val_mae"])
ax[1].plot(ev["epoch"], ev["val_mae"], "o-", label="validation")
if ev["train_mae"].notna().any():
    ax[1].plot(ev["epoch"], ev["train_mae"], "o-", label="train")
ax[1].set(title="MAE liczby kłosów", xlabel="epoka")
ax[1].legend()
plt.show()